In [1]:
import anndata as ad
import h5py
from scipy import sparse
import pandas as pd
import scanpy as sc

In [36]:
m5_adata = ad.read_h5ad("/blue/square.t/peter.huynh/jupyter/combined_samples_scvelo/m345/data/raw/m5_adata.h5ad")

In [37]:
#manually add egfp
f = h5py.File('/blue/square.t/peter.huynh/jupyter/combined_samples_scvelo/m345/data/raw/filtered_feature_bc_matrix.h5', 'r')

barcodes = [b.decode() for b in f['matrix']['barcodes'][:]]
genes = [g.decode() for g in f['matrix']['features']['name'][:]] 

egfp_idx = genes.index("egfp") #a single number

data = f['matrix']['data'][:]
indices = f['matrix']['indices'][:]
indptr = f['matrix']['indptr'][:]
shape = f['matrix']['shape'][:]  
X = sparse.csc_matrix((data, indices, indptr), shape=shape)

egfp_vector = X[egfp_idx, :].toarray().flatten() #(row,column)

formatted_barcodes = ["possorted_genome_bam_UENJ0:" + bc.replace("-1", "x") for bc in barcodes] 
egfp_series = pd.Series(egfp_vector, index=formatted_barcodes)
egfp_vector_aligned = egfp_series.reindex(m5_adata.obs_names).fillna(0).astype(int).values
egfp_var_idx = m5_adata.var_names.get_loc("egfp")
m5_adata.X = m5_adata.X.tolil()
m5_adata.X[:, egfp_var_idx] = egfp_vector_aligned.reshape(-1, 1)
m5_adata.X = m5_adata.X.tocsr()

In [32]:
print((egfp_vector_aligned > 0).sum())

90


In [38]:
##save point
m5_adata_egfp=m5_adata.copy()

In [39]:
adatas = {
    'm5': m5_adata_egfp,
}

for name, adata in adatas.items():
    adata.var_names_make_unique()
    sc.pp.calculate_qc_metrics(adata, inplace=True)

In [40]:
#save this to processed
m5_adata_egfp.write("/blue/square.t/peter.huynh/jupyter/combined_samples_scvelo/m345/data/raw/m5_adata_egfp.h5ad")